# 04 - Inference trên ảnh mới

Dùng checkpoint tốt nhất để dự đoán lớp bệnh cho một ảnh lá. Đây là demo nghiên cứu, không phải công cụ chẩn đoán nông nghiệp.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_transforms
from src.model import build_resnet18

metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
labels = metadata.sort_values('label').drop_duplicates('label')['class_name'].tolist()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_resnet18(len(labels), freeze_backbone=False)
state = torch.load(PROJECT_ROOT / 'checkpoints' / 'E3_resnet_finetune_aug.pt', map_location=device)
model.load_state_dict(state['model_state']); model.to(device).eval()
_, eval_transform = make_transforms(augmentation=False)
IMAGE_PATH = Path(metadata.iloc[0]['path'])  # thay bằng đường dẫn ảnh mới
image = Image.open(IMAGE_PATH).convert('RGB')
with torch.no_grad():
    probabilities = model(eval_transform(image).unsqueeze(0).to(device)).softmax(1)[0]
top_values, top_indices = probabilities.topk(5)
plt.imshow(image); plt.axis('off'); plt.title(labels[top_indices[0]]); plt.show()
[(labels[index], round(float(value), 4)) for value, index in zip(top_values.cpu(), top_indices.cpu())]